In [1]:
%load_ext autoreload
%autoreload 2

Import packages and set up the environment

In [2]:
import os
from dotenv import load_dotenv
from scraper import Website
from IPython.display import Markdown, display
from openai import OpenAI
from openai.types.responses import Response

In [ ]:
from openai.types.chat import ChatCompletionMessageParam
from openai.types.responses import ResponseInputParam
from openai.types.shared_params import Reasoning
# Load environment variables
load_dotenv(override=True) #? override=True means that if the environment variable is already set, it will be overridden.
api_key = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-6-luna"
REASONING: Reasoning = Reasoning(effort="none")

# check the key

if not api_key:
    print("OPENAI_API_KEY is not set")
    raise ValueError("OPENAI_API_KEY is not set")
elif not api_key.startswith("sk-proj-"):
    print("OPENAI_API_KEY is not a valid API key")
    raise ValueError("OPENAI_API_KEY is not a valid API key")
elif api_key.strip() != api_key:
    print("OPENAI_API_KEY contains whitespace")
    raise ValueError("OPENAI_API_KEY contains whitespace")
else:
    print("OPENAI_API_KEY is set and valid")

class InitializeOpenAI:
    def __init__(self):
        self.openai = OpenAI(api_key=api_key)
    def responses(self, input: ResponseInputParam) -> Response:
        return self.openai.responses.create(model=MODEL, reasoning=REASONING, input=input)
    def chat_completions(self, input: list[ChatCompletionMessageParam]):
        return self.openai.chat.completions.create(model=MODEL, messages=input)

OPENAI_API_KEY is set and valid


In [20]:
import requests

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Tell me a fun fact"}]
}

http_response  = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)

res = http_response.json()


In [21]:
res["choices"][0]["message"]["content"]

'Wombats produce cube-shaped poop, which helps keep it from rolling away when they use it to mark their territory.'

Clearner way to do it via the OpenAI API instead of using requests and plain json

In [28]:
chat_response = InitializeOpenAI().chat_completions(input=[{"role": "user", "content": "Tell me a fun fact"}])
chat_response.choices[0].message.content

'Octopuses have three hearts—and two of them stop beating when the octopus swims. That’s one reason they prefer crawling along the seafloor!'

## Google Gemini API

In [36]:
load_dotenv(override=True)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_api_key = os.getenv("GEMINI_API_KEY")
MODEL_GEMINI = "gemini-2.5-flash"

if not gemini_api_key:
    print("GEMINI_API_KEY is not set")
    raise ValueError("GEMINI_API_KEY is not set")
elif not gemini_api_key.startswith("AIza"):
    print("GEMINI_API_KEY is not a valid API key")
    raise ValueError("GEMINI_API_KEY is not a valid API key")
elif gemini_api_key.strip() != gemini_api_key:
    print("GEMINI_API_KEY contains whitespace")
    raise ValueError("GEMINI_API_KEY contains whitespace")
else:
    print("GEMINI_API_KEY is set and valid")

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=gemini_api_key)

gemini_res = gemini.chat.completions.create(model=MODEL_GEMINI, messages=[{"role": "user", "content": "Tell me a fun fact"}])
gemini_res.choices[0].message.content

GEMINI_API_KEY is set and valid


"Here's one!\n\nAn **octopus has three hearts**. Two pump blood through the gills, and the third circulates blood to the rest of the body."

## Ollama

In [39]:
requests.get("http://127.0.0.1:11434").content

b'Ollama is running'

In [44]:
!ollama pull gemma4:e4b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 370c2879f176: 100% ▕██████████████████▏ 5.5 GB                         
pulling 4cb21b935cd7: 100% ▕██████████████████▏ 991 MB                         
pulling 05612e54110f: 100% ▕██████████████████▏  98 MB                         
pulling b507b9c2f6ca: 100% ▕██████████████████▏   13 B                         
pulling 7339fa418c9a: 100% ▕██████████████████▏  11 KB                         
pulling 2365fbb6d97b: 100% ▕██████████████████▏   64 B                         
pulling 7ca1ae564b24: 100% ▕██████████████████▏  241 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success 


In [45]:
OLLAMA_BASE_URL = "http://127.0.0.1:11434/v1"
OLLAMA_MODEL="gemma4:e4b"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [47]:
ollama_res = ollama.chat.completions.create(model=OLLAMA_MODEL, messages=[{"role": "user", "content": "Tell me a fun fact"}])
ollama_res.choices[0].message.content

'Did you know that an octopus has **three hearts**?\n\nTwo of those hearts pump blood through the gills, while the third circulates blood to the rest of the body. They are also incredibly intelligent and have three hearts that are all different! 🐙❤️❤️'

## Summarize with Ollama

In [ ]:

system_prompt = """
tell me about the website and the contents of the website. Responsd in markdown format with heading and bullet points.
"""

user_prompt_prefix = """
Here are the contents of a website:
Keep only the text and remove the emojis and other non-text content.
{contents}
"""

def messages_for(website: Website) -> ResponseInputParam:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix.format(contents=website.contents())}
    ]

rahyab = Website("https://rahyabmohajer.com")

#! chat.completions is the older way to do it and responses is the newer way that needs to be used for the new models with reasoning.
ollama_res = ollama.responses.create(model=OLLAMA_MODEL, input=messages_for(rahyab))

print(ollama_res.output_text)




# 🌐 Website Overview: Rahyab Mohajer

**Website Name:** Rahyab Mohajer | Iranian Migrant Club (رهیاب مهاجر | باشگاه ایرانیان مهاجر)

**Core Purpose:**
*   The website functions as a comprehensive platform ("All in one search") designed to meet all the daily needs of Iranian migrants worldwide.
*   It serves as a community hub and a practical directory, combining business listings, job opportunities, event calendars, and classifieds.

**Key Demographics and Metrics:**
*   **Active Members:** 1,995
*   **Active Categories:** 12
*   **City Coverage:** 19 cities worldwide

***

# 📚 Contents and Features

## 🏢 Business and Service Directory (کسب‌وکارها)
The platform hosts a vast directory of Iranian businesses, providing contact and location information for professionals and services.

*   **Legal and Immigration (حقوقی و مهاجرت):** Services related to legal aid, visas, and residency.
*   **Health and Medical (پزشکی و سلامت):** Listings for doctors, healthcare providers, and nursing service